<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Single Object Detection

Let's find a banana!

In [ ]:
import os
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
import zipfile
from io import BytesIO

In [ ]:
# ---------------------------
# 1) DATA DOWNLOADING & EXTRACTION
# ---------------------------

def download_data(url: str, extract_path: str = "."):
    print("Downloading data...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with zipfile.ZipFile(BytesIO(r.content), "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete.")

The d2l banana-detection dataset is a small, beginner-friendly object detection dataset from Dive into Deep Learning that contains images of bananas annotated with bounding boxes

In [ ]:
# ---------------------------
# 2) DATASET
# ---------------------------
class BananaDataset(Dataset):
    """
    Loads banana images and labels from d2l banana-detection dataset.
    label.csv format: [img_name, label, xmin, ymin, xmax, ymax]
    We return:
      image: Tensor [3,256,256]
      label: Tensor [5] = [class, x1, y1, x2, y2] where bbox is normalized to [0,1]
    """

    def __init__(self, is_train: bool, root_dir: str = "banana-detection", img_size: int = 256):
        super().__init__()
        self.root_dir = root_dir
        self.img_size = img_size

        split_dir = "bananas_train" if is_train else "bananas_val"
        self.data_dir = os.path.join(self.root_dir, split_dir)

        csv_path = os.path.join(self.data_dir, "label.csv")
        if not os.path.exists(csv_path):
            raise FileNotFoundError(
                f"Could not find {csv_path}. Did the zip extract correctly?")

        csv_data = pd.read_csv(csv_path).set_index("img_name")

        self.images = []
        self.labels = []
        for img_name, row in csv_data.iterrows():
            img_path = os.path.join(self.data_dir, "images", img_name)
            self.images.append(img_path)

            # row: [label, xmin, ymin, xmax, ymax]
            row_t = torch.tensor(list(row), dtype=torch.float32)
            cls = row_t[0].view(1)  # typically 0 for banana in this dataset
            bbox = row_t[1:] / float(self.img_size)  # normalize to [0,1]
            self.labels.append(torch.cat([cls, bbox], dim=0))

        self.transform = transforms.Compose(
            [
                transforms.Resize((self.img_size, self.img_size)),
                transforms.ToTensor(),
            ]
        )

    def __getitem__(self, idx: int):
        img = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        return self.transform(img), label

    def __len__(self) -> int:
        return len(self.images)


In [ ]:
# ---------------------------
# 3) MODEL (spatially-aware regressor)
# ---------------------------
class BananaDetector(nn.Module):
    """
    Single-object detector (toy): predicts
      - cls logit (banana vs not banana)
      - bbox [x1,y1,x2,y2] normalized in [0,1]
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(
            ), nn.MaxPool2d(2),   # 256 -> 128
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(
            ), nn.MaxPool2d(2),  # 128 -> 64
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(
            ), nn.MaxPool2d(2),  # 64 -> 32
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(
            ), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8)),  # keep layout (not 1x1)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(256, 1)   # logit
        self.box_regressor = nn.Linear(256, 4)  # raw -> sigmoid -> [0,1]

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        cls_logit = self.classifier(x)
        box = torch.sigmoid(self.box_regressor(x))  # constrain to [0,1]
        # enforce ordering: (x1,y1) = min, (x2,y2) = max
        x1y1 = torch.minimum(box[:, :2], box[:, 2:])
        x2y2 = torch.maximum(box[:, :2], box[:, 2:])
        box = torch.cat([x1y1, x2y2], dim=1)
        return cls_logit, box


In [ ]:
# ---------------------------
# 4) VISUALIZATION UTILITIES
# ---------------------------
def show_bbox(img_tensor: torch.Tensor, bbox_norm: torch.Tensor, title: str, img_size: int = 256):
    """
    img_tensor: [3,H,W] (0..1)
    bbox_norm: [4] normalized [0,1] => converted to pixels for plotting
    """
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    bbox = (bbox_norm.detach().cpu() * img_size).clamp(0, img_size)

    x1, y1, x2, y2 = bbox.tolist()
    w, h = max(1.0, x2 - x1), max(1.0, y2 - y1)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    rect = patches.Rectangle((x1, y1), w, h, linewidth=2,
                             edgecolor="r", facecolor="none")
    plt.gca().add_patch(rect)
    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
# ---------------------------
# 5) TRAIN / EVAL
# ---------------------------
def train_one_epoch(model, loader, optimizer, criterion_cls, criterion_box, device, cls_weight=0.2, box_weight=1.0):
    model.train()
    running = 0.0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        # All images contain banana in this dataset (no negatives)
        target_cls = torch.ones(
            (labels.shape[0], 1), device=device)  # banana=1
        target_box = labels[:, 1:]  # [x1,y1,x2,y2]

        pred_cls, pred_box = model(imgs)

        loss_cls = criterion_cls(pred_cls, target_cls)
        loss_box = criterion_box(pred_box, target_box)
        loss = cls_weight * loss_cls + box_weight * loss_box

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running += loss.item()

    return running / max(1, len(loader))

In [ ]:
@torch.no_grad()
def eval_sample(model, dataset, idx, device):
    model.eval()
    img, label = dataset[idx]
    img_b = img.unsqueeze(0).to(device)
    cls_logit, pred_box = model(img_b)
    pred_box = pred_box.squeeze(0).cpu().clamp(0, 1)
    gt_box = label[1:].cpu()
    return img.cpu(), gt_box, pred_box, torch.sigmoid(cls_logit).item()


Note: Ground truth (GT) is the correct, trusted reference label for a dataset: what the model is supposed to learn to predict. It’s usually produced by human annotation or a reliable measurement process, and it’s used to compute the loss during training and to evaluate performance.

In [ ]:
# ---------------------------
# 6) MAIN
# ---------------------------
def main():
    # Reproducibility
    torch.manual_seed(0)

    # Download dataset if needed
    url = "http://d2l-data.s3-accelerate.amazonaws.com/banana-detection.zip"
    if not os.path.exists("banana-detection"):
        download_data(url)

    # Data
    train_set = BananaDataset(is_train=True)
    val_set = BananaDataset(is_train=False)
    train_loader = DataLoader(train_set, batch_size=32,
                              shuffle=True, num_workers=0, pin_memory=False)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Model
    model = BananaDetector().to(device)

    # Optim / Loss
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion_cls = nn.BCEWithLogitsLoss()
    # often more stable than plain L1
    criterion_box = nn.SmoothL1Loss(beta=0.05)

    epochs = 30
    history = []

    # Quick sanity check: visualize a GT box
    img0, lab0 = train_set[0]
    show_bbox(img0, lab0[1:], "GT sanity check")

    print("Starting training...")
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(
            model, train_loader, optimizer, criterion_cls, criterion_box, device,
            cls_weight=0.2, box_weight=1.0
        )
        history.append(loss)
        print(f"Epoch [{epoch:02d}/{epochs}], Loss: {loss:.4f}")

    # Plot training curve
    plt.figure()
    plt.plot(history)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.show()

    # Evaluate a few samples
    for idx in [0, 8, 20]:
        img, gt_box, pred_box, p = eval_sample(model, val_set, idx, device)
        show_bbox(img, gt_box, f"VAL idx={idx} | GT")
        show_bbox(img, pred_box, f"VAL idx={idx} | Pred (p={p:.2f})")

    # Also show a train sample prediction (for comparison)
    img, gt_box, pred_box, p = eval_sample(model, train_set, 8, device)
    show_bbox(img, gt_box, "TRAIN idx=8 | GT")
    show_bbox(img, pred_box, f"TRAIN idx=8 | Pred (p={p:.2f})")

Let's see the detection in action:

In [ ]:
if __name__ == "__main__":
    main()